# **Koneksi Ke Google Big Query**

In [1]:
from google.colab import auth, data_table
from google.cloud import bigquery
from pandas_gbq import to_gbq
import pandas as pd
import numpy as np

# PRoses autentikasi akun
auth.authenticate_user()
print('Authenticated')

Authenticated


In [2]:
# Buat BigQuery client
project_id = 'etl-4414'
client = bigquery.Client(project = project_id)

# **Proses ETL**

## **Proses 1 - tbl_transaction to tbl_dwh_transaction**
Lakukan :
- Ekstraksi data dari sumber (_extract_)
- Inspeksi informasi umum pada data
- Transformasikan data jika diperlukan
- Cleansing data jika diperlukan
- Tambahkan metadata

### Proses ekstraksi data tbl_transaction

In [3]:
# Inisialisasi query yang akan dijalankan
query = "SELECT * FROM dqlab-468906.dqcommerce.tbl_transaction"

# Proses ekstraksi data dari BigQuery ke pandas
data_transaksi = client.query(query).to_dataframe()

# Tampilkan hasil
data_transaksi.sample(5)


,trx_id,product_id,trx_date,trx_time,units
544500,DQTrx430483,DQProduk-025,7012025,06:38:31,1
234440,DQTrx531786,DQProduk-008,19032025,12:39:35,1
664775,DQTrx806379,DQProduk-031,15092025,16:01:06,1
231708,DQTrx484909,DQProduk-008,15022025,06:18:34,1
785087,DQTrx489251,DQProduk-019,18022025,07:22:09,3


### Inspeksi informasi umum pada data tbl_transaction


In [4]:
# Informasi umum pada data
data_transaksi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 829262 entries, 0 to 829261
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   trx_id      829257 non-null  object
 1   product_id  829262 non-null  object
 2   trx_date    829262 non-null  Int64 
 3   trx_time    829262 non-null  dbtime
 4   units       829251 non-null  Int64 
dtypes: Int64(2), dbtime(1), object(2)
memory usage: 33.2+ MB


Temuan :     
- Jumlah data : 829262 baris dengan 5 kolom
- Terdapat missing value pada kolom trx_id dan units
- Kolom trx_date memiliki tipe data yang bisa disesuaikan

In [16]:
#Missing value pada kolom trx_id
miss_trx_id = data_transaksi['trx_id'].isnull()
data_transaksi[miss_trx_id]

,trx_id,product_id,trx_date,trx_time,units
106825,None,DQProduk-004,1012024,14:43:09,1
193154,None,DQProduk-008,1012024,09:27:56,1
468999,None,DQProduk-021,1012024,13:16:09,1
499044,None,DQProduk-024,1012024,06:32:49,1
571924,None,DQProduk-027,1012024,04:15:36,1


In [17]:
#Missing value pada kolom units
miss_units = data_transaksi['units'].isnull()
print(data_transaksi[miss_units])

         trx_id    product_id  trx_date  trx_time  units
0   DQTrx829214  DQProduk-016  30092025  01:18:05   <NA>
1   DQTrx829166  DQProduk-017  30092025  18:58:33   <NA>
2   DQTrx829118  DQProduk-018  30092025  06:58:09   <NA>
3   DQTrx829186  DQProduk-019  30092025  09:28:13   <NA>
4   DQTrx829212  DQProduk-019  30092025  15:00:55   <NA>
5   DQTrx829213  DQProduk-019  30092025  16:06:16   <NA>
6   DQTrx829235  DQProduk-019  30092025  17:49:03   <NA>
7   DQTrx829167  DQProduk-024  30092025  21:51:24   <NA>
8   DQTrx829168  DQProduk-025  30092025  02:05:10   <NA>
9   DQTrx829254  DQProduk-025  30092025  08:12:00   <NA>
10  DQTrx829169  DQProduk-028  30092025  06:03:36   <NA>


### Proses cleansing data tbl_transaction

- Hapus baris jika trx_id nya kosong
- Isikan 0 pada units yang kosong
- Hapus duplikasi data

In [18]:
# Hapus baris jika trx_idnya kosong
data_transaksi = data_transaksi.dropna(subset = ['trx_id'])

# Isikan 0 pada kolom units yang kosong
data_transaksi['units'] = data_transaksi['units'].fillna(0)

# Hapus duplikasi data
data_transaksi = data_transaksi.drop_duplicates()

# Tampilkan hasil cleansing data
data_transaksi.info()

<class 'pandas.core.frame.DataFrame'>
Index: 829257 entries, 0 to 829261
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   trx_id      829257 non-null  object
 1   product_id  829257 non-null  object
 2   trx_date    829257 non-null  Int64 
 3   trx_time    829257 non-null  dbtime
 4   units       829257 non-null  Int64 
dtypes: Int64(2), dbtime(1), object(2)
memory usage: 39.5+ MB


### Proses transformasi data tbl_transaction

- Ubah format tanggal menjadi ISO 8601 yakni YYYY-MM-DD (https://www.iso.org/iso-8601-date-and-time-format.html)


In [22]:
#tabel trx_date sebelum transformasi
data_transaksi['trx_date'] = '0' + data_transaksi['trx_date'].astype(str)
data_transaksi.sample(5)

,trx_id,product_id,trx_date,trx_time,units
119638,DQTrx043512,DQProduk-006,00014022024,07:43:55,1
309097,DQTrx618659,DQProduk-012,00012052025,20:15:28,1
740356,DQTrx179319,DQProduk-031,00015062024,05:44:55,2
99722,DQTrx710638,DQProduk-003,0008072025,05:12:47,1
787146,DQTrx647702,DQProduk-019,00030052025,06:53:54,3


In [23]:
# Tambahkan 0 di awal
data_transaksi['trx_date'] = '0' + data_transaksi['trx_date'].astype(str)

# Ambil 8 karakter dari kanan
data_transaksi['trx_date'] = data_transaksi['trx_date'].str[-8:]

# Ubah menjadi datetime
data_transaksi['trx_date'] = pd.to_datetime(data_transaksi['trx_date'], format = '%d%m%Y')

# Tampilkan hasilnya
data_transaksi.sample(5)

,trx_id,product_id,trx_date,trx_time,units
268461,DQTrx683482,DQProduk-009,2025-06-22,09:30:04,1
461973,DQTrx759767,DQProduk-019,2025-08-10,21:50:50,1
259007,DQTrx392610,DQProduk-009,2024-12-14,07:13:25,1
210093,DQTrx200518,DQProduk-008,2024-07-02,02:20:08,1
804724,DQTrx109054,DQProduk-031,2024-04-17,20:37:03,3


### Tambahkan Metadata

- Tambahkan kolom insert by lalu isikan dengan SYSTEM
- Tambahkan kolom insert date lalu isikan dengan tanggal dan waktu sekarang

In [25]:
# Import library
from datetime import datetime

# Tambahkan metadata
data_transaksi['insert_by'] = 'SYSTEM'
data_transaksi['insert_date'] = datetime.now()

# Tampilkan hasilnya
data_transaksi.sample(5)

,trx_id,product_id,trx_date,trx_time,units,insert_by,insert_date
330407,DQTrx102818,DQProduk-014,2024-04-12,03:27:30,1,SYSTEM,2025-10-08 14:48:02.604637
226173,DQTrx414118,DQProduk-008,2024-12-27,18:00:07,1,SYSTEM,2025-10-08 14:48:02.604637
487413,DQTrx787715,DQProduk-021,2025-09-01,04:17:30,1,SYSTEM,2025-10-08 14:48:02.604637
357826,DQTrx504011,DQProduk-015,2025-03-01,03:39:08,1,SYSTEM,2025-10-08 14:48:02.604637
468855,DQTrx749593,DQProduk-020,2025-08-03,23:08:39,1,SYSTEM,2025-10-08 14:48:02.604637


### Insert ke Google Big Query

docs : _https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_gbq.html_

In [26]:
# Definisikan tabel schema
table_schema_tbl_transaction = [
    {'name': 'trx_id', 'type': 'STRING'},
    {'name': 'product_id', 'type': 'STRING'},
    {'name': 'trx_date', 'type': 'DATE'},
    {'name': 'trx_time', 'type': 'TIME'},
    {'name': 'units', 'type': 'INT64'},
    {'name': 'insert_by', 'type': 'STRING'},
    {'name': 'insert_date', 'type': 'TIMESTAMP'}
]

# Upload ke BigQuery
to_gbq(
    data_transaksi,
    destination_table = 'data_warehouse.tbl_dwh_transaction',
    project_id = project_id,
    if_exists = 'append',
    table_schema = table_schema_tbl_transaction
)

100%|██████████| 1/1 [00:00<00:00, 9098.27it/s]


## **Proses 2 - tbl_product to tbl_dwh_product**
Lakukan :
- Ekstraksi data dari sumber (_extract_)
- Inspeksi informasi umum pada data
- Transformasikan data jika diperlukan
- Cleansing data jika diperlukan
- Tambahkan metadata

### Proses ekstraksi data tbl_product

In [27]:
# Inisialisasi query yang akan dijalankan
query = "SELECT * FROM dqlab-468906.dqcommerce.tbl_product"

# Proses ekstraksi data dari BigQuery ke pandas
data_product = client.query(query).to_dataframe()
data_product.sample(5)

,product_id,product_name,product_category,product_cost,product_price
34,DQProduk-020,Papan Klip DQLab,Perlengkapan Meja,IDR 134850,IDR 374850
7,DQProduk-021,Kalkulator Besar DQLab,Elektronik,IDR 104850,IDR 149850
23,DQProduk-002,Termos Logo DQLab,Merchandise,IDR 149850,IDR 194850
3,DQProduk-016,Set Pulpen DQLab,Alat Tulis,IDR 44850,IDR 149850
19,DQProduk-014,Label Nama DQLab,Kertas & Cetak,IDR 89850,IDR 164850


### Inspeksi informasi umum pada data tbl_product


In [28]:
# Informasi umum pada data
data_product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   product_id        37 non-null     object
 1   product_name      37 non-null     object
 2   product_category  37 non-null     object
 3   product_cost      37 non-null     object
 4   product_price     37 non-null     object
dtypes: object(5)
memory usage: 1.6+ KB


Temuan :     
- Jumlah data : 37 baris dengan 5 kolom
- Tidak terdapat missing value pada semua kolom
- Kolom product_cost dan product_price memiliki tipe data yang bisa disesuaikan

### Proses cleansing data tbl_product

- Hapus baris jika product_id nya kosong
- Hapus duplikasi data
- Hapus currency pada kolom product_cost dan product_price

In [40]:
#mengecek data_product yang terdapat duplikasi sebelum di hilangkan
data_product[data_product.duplicated(keep = False)]

,product_id,product_name,product_category,product_cost,product_price
17,DQProduk-035,Kartu Ucapan DQLab,Kertas & Cetak,IDR 59850,IDR 119850
18,DQProduk-035,Kartu Ucapan DQLab,Kertas & Cetak,IDR 59850,IDR 119850
27,DQProduk-015,Mousepad Custom DQLab,Merchandise,IDR 59850,IDR 89850
28,DQProduk-015,Mousepad Custom DQLab,Merchandise,IDR 59850,IDR 89850


In [41]:
# Hapus baris jika product_id nya kosong
data_product = data_product.dropna(subset = ['product_id'])

# Hapus duplikasi data
data_product = data_product.drop_duplicates()

# Hapus currency pada kolom product_cost dan product_price
data_product['product_cost'] = data_product['product_cost'].str.replace('IDR ', '')
data_product['product_price'] = data_product['product_price'].str.replace('IDR ', '')

# Tampilkan hasilnya
data_product.head()

,product_id,product_name,product_category,product_cost,product_price
0,DQProduk-009,Pulpen Metal DQLab,Alat Tulis,149850,164850
1,DQProduk-003,Sticky Notes DQLab,Alat Tulis,29850,59850
2,DQProduk-025,Tipe-X Roller DQLab,Alat Tulis,29850,44850
3,DQProduk-016,Set Pulpen DQLab,Alat Tulis,44850,149850
4,DQProduk-028,Penanda Warna DQLab,Alat Tulis,59850,164850


### Proses transformasi data tbl_product

- Ubah format tanggal menjadi ISO 8601 yakni YYYY-MM-DD (https://www.iso.org/iso-8601-date-and-time-format.html)

In [42]:
data_product['product_cost'] = data_product['product_cost'].astype(pd.Float64Dtype())
data_product['product_price'] = data_product['product_price'].astype(pd.Float64Dtype())

In [45]:
#Cek informasi umum pada data setelah menghilangkan duplikasi
data_product.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35 entries, 0 to 36
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   product_id        35 non-null     object        
 1   product_name      35 non-null     object        
 2   product_category  35 non-null     object        
 3   product_cost      35 non-null     Float64       
 4   product_price     35 non-null     Float64       
 5   insert_by         35 non-null     object        
 6   insert_date       35 non-null     datetime64[us]
dtypes: Float64(2), datetime64[us](1), object(4)
memory usage: 2.3+ KB


### Tambahkan Metadata

- Tambahkan kolom insert by lalu isikan dengan SYSTEM
- Tambahkan kolom insert date lalu isikan dengan tanggal dan waktu sekarang

In [43]:
# Import library
from datetime import datetime

# Tambahkan metadata
data_product['insert_by'] = 'SYSTEM'
data_product['insert_date'] = datetime.now()

# Tampilkan hasilnya
data_product.head()

,product_id,product_name,product_category,product_cost,product_price,insert_by,insert_date
0,DQProduk-009,Pulpen Metal DQLab,Alat Tulis,149850.0,164850.0,SYSTEM,2025-10-08 15:10:18.715169
1,DQProduk-003,Sticky Notes DQLab,Alat Tulis,29850.0,59850.0,SYSTEM,2025-10-08 15:10:18.715169
2,DQProduk-025,Tipe-X Roller DQLab,Alat Tulis,29850.0,44850.0,SYSTEM,2025-10-08 15:10:18.715169
3,DQProduk-016,Set Pulpen DQLab,Alat Tulis,44850.0,149850.0,SYSTEM,2025-10-08 15:10:18.715169
4,DQProduk-028,Penanda Warna DQLab,Alat Tulis,59850.0,164850.0,SYSTEM,2025-10-08 15:10:18.715169


### Insert ke Google Big Query

docs : _https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_gbq.html_

In [46]:
# Definisikan tabel schema
table_schema_tbl_product = [
    {'name': 'product_id', 'type': 'STRING'},
    {'name': 'product_name', 'type': 'STRING'},
    {'name': 'product_category', 'type': 'STRING'},
    {'name': 'product_cost', 'type': 'FLOAT'},
    {'name': 'product_price', 'type': 'FLOAT'},
    {'name': 'insert_by', 'type': 'STRING'},
    {'name': 'insert_date', 'type': 'TIMESTAMP'}
]

# Upload ke BigQuery
to_gbq(
    data_product,
    destination_table = 'data_warehouse.tbl_dwh_product',
    project_id = project_id,
    if_exists = 'append',
    table_schema = table_schema_tbl_product
)

100%|██████████| 1/1 [00:00<00:00, 1724.63it/s]


## **Proses 3 - tbl_funnel to tbl_dwh_funnel**
Lakukan :
- Ekstraksi data dari sumber (_extract_)
- Inspeksi informasi umum pada data
- Transformasikan data jika diperlukan
- Cleansing data jika diperlukan
- Tambahkan metadata

### Proses ekstraksi data tbl_funnel

In [47]:
# Inisialisasi query yang akan dijalankan
query = "SELECT * FROM dqlab-468906.dqcommerce.tbl_funnel"

# Proses ekstraksi data dari BigQuery ke pandas
data_funnel = client.query(query).to_dataframe()
data_funnel.sample(5)

,date,product_id,purchase,add_to_cart,click,view
9912,13042025,DQProduk-021,58,541,1706,2317
14479,19052025,DQProduk-032,1,10,43,56
5928,27092025,DQProduk-013,34,281,1090,2106
8374,10032024,DQProduk-017,99,1043,5902,11570
4242,21082025,DQProduk-009,18,151,551,945


### Inspeksi informasi umum pada data tbl_funnel


In [48]:
# Informasi umum pada data
data_funnel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15950 entries, 0 to 15949
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         15950 non-null  Int64 
 1   product_id   15950 non-null  object
 2   purchase     15950 non-null  Int64 
 3   add_to_cart  15950 non-null  Int64 
 4   click        15950 non-null  Int64 
 5   view         15950 non-null  Int64 
dtypes: Int64(5), object(1)
memory usage: 825.7+ KB


Temuan :     
- Jumlah data : 15950 baris dengan 6 kolom
- Tidak terdapat missing value pada semua kolom
- Kolom date memiliki tipe data yang bisa disesuaikan

### Proses cleansing data tbl_product

- Hapus baris jika product_id nya kosong
- Hapus duplikasi data

In [49]:
# Hapus baris jika product_idnya kosong
data_funnel = data_funnel.dropna(subset = ['product_id'])

# Hapus duplikasi data
data_funnel = data_funnel.drop_duplicates()

# Tampilkan hasil cleansing data
data_funnel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15950 entries, 0 to 15949
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         15950 non-null  Int64 
 1   product_id   15950 non-null  object
 2   purchase     15950 non-null  Int64 
 3   add_to_cart  15950 non-null  Int64 
 4   click        15950 non-null  Int64 
 5   view         15950 non-null  Int64 
dtypes: Int64(5), object(1)
memory usage: 825.7+ KB


### Proses transformasi data tbl_funnel

- Ubah format tanggal menjadi ISO 8601 yakni YYYY-MM-DD (https://www.iso.org/iso-8601-date-and-time-format.html)


In [50]:
# Tambahkan 0 di awal
data_funnel['date'] = '0' + data_funnel['date'].astype(str)

# Ambil 8 karakter dari kanan
data_funnel['date'] = data_funnel['date'].str[-8:]

# Ubah menjadi datetime
data_funnel['date'] = pd.to_datetime(data_funnel['date'], format = '%d%m%Y')

# Tampilkan hasilnya
data_funnel.head()

,date,product_id,purchase,add_to_cart,click,view
0,2025-07-16,DQProduk-001,3,34,194,253
1,2025-08-07,DQProduk-001,3,32,184,280
2,2025-07-17,DQProduk-001,5,57,220,263
3,2025-08-22,DQProduk-001,5,51,196,302
4,2024-10-05,DQProduk-001,7,82,298,589


### Tambahkan Metadata

- Tambahkan kolom insert by lalu isikan dengan SYSTEM
- Tambahkan kolom insert date lalu isikan dengan tanggal dan waktu sekarang

In [51]:
# Import library
from datetime import datetime

# Tambahkan metadata
data_funnel['insert_by'] = 'SYSTEM'
data_funnel['insert_date'] = datetime.now()

# Tampilkan hasilnya
data_funnel.head()

,date,product_id,purchase,add_to_cart,click,view,insert_by,insert_date
0,2025-07-16,DQProduk-001,3,34,194,253,SYSTEM,2025-10-08 15:22:45.126743
1,2025-08-07,DQProduk-001,3,32,184,280,SYSTEM,2025-10-08 15:22:45.126743
2,2025-07-17,DQProduk-001,5,57,220,263,SYSTEM,2025-10-08 15:22:45.126743
3,2025-08-22,DQProduk-001,5,51,196,302,SYSTEM,2025-10-08 15:22:45.126743
4,2024-10-05,DQProduk-001,7,82,298,589,SYSTEM,2025-10-08 15:22:45.126743


### Insert ke Google Big Query

docs : _https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_gbq.html_

In [52]:
# Definisikan tabel schema
table_schema_tbl_funnel = [
    {'name': 'date', 'type': 'DATE'},
    {'name': 'product_id', 'type': 'STRING'},
    {'name': 'purchase', 'type': 'INT64'},
    {'name': 'add_to_cart', 'type': 'INT64'},
    {'name': 'click', 'type': 'INT64'},
    {'name': 'view', 'type': 'INT64'},
    {'name': 'insert_by', 'type': 'STRING'},
    {'name': 'insert_date', 'type': 'TIMESTAMP'}
]

# Upload ke BigQuery
to_gbq(
    data_funnel,
    destination_table = 'data_warehouse.tbl_dwh_funnel',
    project_id = project_id,
    if_exists = 'append',
    table_schema = table_schema_tbl_funnel
)

100%|██████████| 1/1 [00:00<00:00, 10230.01it/s]


## **Proses Akhir - Data Mart**

In [57]:
query = """
WITH transaction AS (
   SELECT
      product_id,
      trx_date,
      SUM(units) AS total_unit
   FROM etl-4414.data_warehouse.tbl_dwh_transaction
   GROUP BY product_id, trx_date
),
product AS (
    SELECT DISTINCT
      product_id,
      product_cost,
     product_price
   FROM etl-4414.data_warehouse.tbl_dwh_product
),
funnels AS (
    SELECT DISTINCT
      `date`,
      product_id,
      SUM(purchase) AS total_purchase,
      SUM(add_to_cart) AS total_add_to_cart,
      SUM(click) AS total_click,
      SUM(view) AS total_view
   FROM etl-4414.data_warehouse.tbl_dwh_funnel
   GROUP BY `date`, product_id
)
SELECT
      t.product_id,
      t.trx_date,
      t.total_unit,
      p.product_cost AS cost_each,
      p.product_price AS price_each,
      t.total_unit * p.product_price AS total_price,
      t.total_unit * (p.product_price * p.product_cost) AS total_profit,
      f.total_purchase,
      f.total_add_to_cart,
      f.total_click,
      f.total_view,
      'SYSTEM' AS insert_by,
      CURRENT_DATETIME() AS insert_date
FROM funnels AS f
LEFT JOIN transaction AS t ON t.product_id = f.product_id AND t.trx_date = f.date
INNER JOIN product AS p ON t.product_id = p.product_id
"""

# Proses ekstraksi data dari BigQuery ke pandas
data_mart = client.query(query).to_dataframe()
data_mart.sample(5)

,product_id,trx_date,total_unit,cost_each,price_each,total_price,total_profit,total_purchase,total_add_to_cart,total_click,total_view,insert_by,insert_date
15348,DQProduk-030,2024-09-29,63,269850.0,299850.0,18890550.0,5.097615e+12,63,720,2970,3102,SYSTEM,2025-10-08 15:49:23.786684
13143,DQProduk-011,2024-12-01,43,164850.0,314850.0,13538550.0,2.231830e+12,43,430,2210,2414,SYSTEM,2025-10-08 15:49:23.786684
12869,DQProduk-009,2025-09-06,41,149850.0,164850.0,6758850.0,1.012814e+12,41,340,1871,2728,SYSTEM,2025-10-08 15:49:23.786684
1768,DQProduk-008,2024-04-06,169,59850.0,104850.0,17719650.0,1.060521e+12,169,1424,4332,7941,SYSTEM,2025-10-08 15:49:23.786684
5771,DQProduk-029,2025-01-28,4,134850.0,299850.0,1199400.0,1.617391e+11,4,46,141,182,SYSTEM,2025-10-08 15:49:23.786684


In [58]:
# Definisikan tabel schema
table_schema_datamart = [
    {'name': 'product_id', 'type': 'STRING'},
    {'name': 'trx_date', 'type': 'DATE'},
    {'name': 'total_unit', 'type': 'INT64'},
    {'name': 'cost_each', 'type': 'FLOAT'},
    {'name': 'price_each', 'type': 'FLOAT'},
    {'name': 'total_price', 'type': 'FLOAT'},
    {'name': 'total_profit', 'type': 'FLOAT'},
    {'name': 'total_purchase', 'type': 'INT64'},
    {'name': 'total_add_to_cart', 'type': 'INT64'},
    {'name': 'total_click', 'type': 'INT64'},
    {'name': 'total_view', 'type': 'INT64'},
    {'name': 'insert_by', 'type': 'STRING'},
    {'name': 'insert_date', 'type': 'TIMESTAMP'}
]

# Upload ke BigQuery
to_gbq(
    data_mart,
    destination_table = 'data_mart.f_summary_transaction',
    project_id = project_id,
    if_exists = 'append',
    table_schema = table_schema_datamart
)

100%|██████████| 1/1 [00:00<00:00, 11397.57it/s]


## **Open access bagi user (GRANT)**